[Source](https://docs.nvidia.com/bionemo-framework/1.10/notebooks/MolMIM_GenerativeAI_local_inference_with_examples.html)

In [ ]:
%%capture --no-display --no-stderr cell_output

import os

# RDKit for handling/manipulating chemical data
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

import matplotlib.pyplot as plt

from bionemo.utils.hydra import load_model_config
from bionemo.model.molecule.molmim.infer import MolMIMInference

import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import logging
logging.basicConfig(level = logging.INFO)
logging.getLogger("nemo_logger").setLevel(logging.ERROR)

In [ ]:
bionemo_home = "/workspace/bionemo"
os.environ['BIONEMO_HOME'] = bionemo_home
os.chdir(bionemo_home)

In [ ]:
%%capture --no-display cell_output
# !python download_artifacts.py --model_dir ${BIONEMO_HOME}/models --models molmim_70m_24_3

In [ ]:
# %%capture --no-display --no-stderr cell_output

# Load pre-trained model checkpoints
checkpoint_path = f"{bionemo_home}/models/molecule/molmim/molmim_70m_24_3.nemo"

# Load starting config for MolMIM inference
cfg = load_model_config(config_name="molmim_infer.yaml", config_path=f"{bionemo_home}/examples/tests/conf/")

# Point YAML configuration file to the location of the desired checkpoints
cfg.model.downstream_task.restore_from_path = checkpoint_path
#cfg.model.encoder.hidden_steps = 2

# Create model object based on desired configuration
model = MolMIMInference(cfg, interactive=True)

In [ ]:
# Two SMILES strings
smis = ['C#CC(C=C1)=CC=C1C#N','CC(C)C(C=C1)=CC=C1N2C(C=C(C3=CC=C(C#N)C=C3)N4C5=CC=C(C(C)C)C=C5)=C4C=C2C6=CC=C(C#N)C=C6']

# RDKit's MolFromSmiles() function displays molecule from the SMILES string
m1 = Chem.MolFromSmiles(smis[0])
m2 = Chem.MolFromSmiles(smis[1])
Draw.MolsToGridImage((m1,m2), legends=["Smiles 1","Smiles 2"], subImgSize=(300,200))

In [ ]:
# obtaining the hidden state representations for input SMILES
hidden_states, pad_masks = model.seq_to_hiddens(smis)
hidden_states.shape, pad_masks.shape

In [ ]:
embedding = model.seq_to_embeddings(smis)
embedding.shape

In [ ]:
# Obtaining SMILES chemical representation from a hidden state
inferred_smis = model.hiddens_to_seq(hidden_states, pad_masks)

print("Inferred SMILES: ", inferred_smis)

inf_1 = Chem.MolFromSmiles(inferred_smis[0])
inf_2 = Chem.MolFromSmiles(inferred_smis[1])

print(inf_1, inf_2)

Draw.MolsToGridImage((inf_1,inf_2),legends=["Inferred Compound 1","Inferred Compound 2"], subImgSize=(350,350))

In [ ]:
print("Compund 1:", smis[0]==inferred_smis[0])
print("Compund 2:", smis[1]==inferred_smis[1])

In [ ]:
def chem_sample(smis):
    # PART 1: SAMPLING
    # 1A: Set Sampling Arguments
    num_samples = 10
    scaled_radius = 0.7
    sampling_method="beam-search-perturbate" # Options: greedy-perturbate, topkp-perturbate, beam-search-perturbate, beam-search-single-sample, beam-search-perturbate-sample
    sampler_kwargs = {"beam_size": 3, "keep_only_best_tokens": True, "return_scores": False}
    # 1B: Execute sampling
    population_samples = model.sample(seqs=smis, num_samples=num_samples, scaled_radius=scaled_radius,
                                      sampling_method="beam-search-perturbate", **sampler_kwargs)

    # PART 2: FILTERING
    uniq_canonical_smiles = []
    # Loop through each seed molecule
    for smis_samples, original in zip(population_samples, smis):
        # 2A: Gather unique strings (remove duplicates and starting string)
        smis_samples = set(smis_samples) - set([original])

        # 2B: Validate generated molecules
        valid_molecules = []
        for smis in smis_samples:
            mol = Chem.MolFromSmiles(smis)
            if mol:
                valid_molecules.append(Chem.MolToSmiles(mol,True))
        uniq_canonical_smiles.append(valid_molecules)
    return uniq_canonical_smiles # List (len = seed compounds) of lists (len = generated compounds per seed)

In [ ]:
# Redefine inputs and generate new molecules
ori_smis_lst = ['C#CC(C=C1)=CC=C1C#N','CC(C)C(C=C1)=CC=C1N2C(C=C(C3=CC=C(C#N)C=C3)N4C5=CC=C(C(C)C)C=C5)=C4C=C2C6=CC=C(C#N)C=C6']
gen_smis_lst = chem_sample(ori_smis_lst)

for ori_smis, gen_smis in zip(ori_smis_lst, gen_smis_lst):
        print(f"Original SMILES: {ori_smis}")
        print(f"Generated {len(gen_smis)} unique/valid SMILES: {gen_smis}")
        print("\n")

## C#CC(C=C1)=CC=C1C#N

In [ ]:
m1 = Chem.MolFromSmiles('C#CC(C=C1)=CC=C1C#N')
Draw.MolToImage(m1)

Generated molecules:

In [ ]:
mols_from_gen_smis = [Chem.MolFromSmiles(smi) for smi in set(gen_smis_lst[0])]
print("Total unique molecule designs obtained: ", len(mols_from_gen_smis))
Draw.MolsToGridImage(mols_from_gen_smis, molsPerRow=5, subImgSize=(300,300))

## CC(C)C(C=C1)=CC=C1N2C(C=C(C3=CC=C(C#N)C=C3)N4C5=CC=C(C(C)C)C=C5)=C4C=C2C6=CC=C(C#N)C=C6

In [ ]:
m2 = Chem.MolFromSmiles('CC(C)C(C=C1)=CC=C1N2C(C=C(C3=CC=C(C#N)C=C3)N4C5=CC=C(C(C)C)C=C5)=C4C=C2C6=CC=C(C#N)C=C6')
Draw.MolToImage(m2)

Generated molecules:

In [ ]:
mols_from_gen_smis = [Chem.MolFromSmiles(smi) for smi in set(gen_smis_lst[1])]
print("Total unique molecule designs obtained: ", len(mols_from_gen_smis))
Draw.MolsToGridImage(mols_from_gen_smis, molsPerRow=5, subImgSize=(300,300))

# Prediction using MolMIM embeddings

TODO: [Downstream Prediction Model Using Learned Embeddings from MolMIM](https://docs.nvidia.com/bionemo-framework/1.10/notebooks/MolMIM_GenerativeAI_local_inference_with_examples.html#downstream-prediction-model-using-learned-embeddings-from-molmim)

In [ ]:
ex_data_file = f"{bionemo_home}/data/data_experts_1.csv"
ex_df = pd.read_csv(ex_data_file)
print(ex_df.shape)
ex_df.head()

In [ ]:
ex_emb_df = pd.DataFrame()
smis = ex_df.loc[:, 'smiles']
esol = ex_df.loc[:, 'capacity_max']
embedding = model.seq_to_embeddings(smis.tolist())
ex_emb_df = pd.concat([ex_emb_df,
                       pd.DataFrame({"SMILES": smis,
                                     "EMBEDDINGS": embedding.tolist(),
                                     "Y": esol})])

ex_emb_df

In [ ]:
# Splitting dataset into training and testing sets
tempX = np.asarray(ex_emb_df['EMBEDDINGS'].tolist(), dtype=np.float32)
tempY = np.asarray(ex_emb_df['Y'], dtype=np.float32)
x_train, x_test_emb, y_train, y_test_emb  = train_test_split(tempX, tempY, train_size=0.7, random_state=1993)

# Defining SVR model parameters (you may change them and observe change in performance)
reg_emb = SVR(kernel='rbf', gamma='scale', C=10, epsilon=0.01)

# Fitting the model on the training dataset
reg_emb.fit(x_train, y_train)

# Using the fitted model for prediction of the ESOL values on test dataset
pred_emb = reg_emb.predict(x_test_emb)

# Performance measures of SVR model
emb_SVR_MSE = mean_squared_error(y_test_emb, pred_emb)
emb_SVR_R2 = r2_score(y_test_emb, pred_emb)

print("Embeddings_SVR_MSE: ", emb_SVR_MSE)
print("Embeddings_SVR_r2: ", emb_SVR_R2)

In [ ]:
# Create the scatter plot
plt.figure(figsize=(10, 8))
plt.scatter(y_test_emb, pred_emb, alpha=0.5)

# Add a diagonal line representing perfect predictions
min_val = min(min(y_test_emb), min(pred_emb))
max_val = max(max(y_test_emb), max(pred_emb))
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)

# Set labels and title
plt.xlabel('Actual capacity_max Values')
plt.ylabel('Predicted capacity_max Values')
plt.title('Predicted vs. Actual capacity_max Values from SVR Model Trained on MolMIM Embeddings')

# Add R-squared value to the plot
plt.text(0.05, 0.95, f'R² = {emb_SVR_R2:.4f}', transform=plt.gca().transAxes,
         verticalalignment='top')

# Adjust layout and display the plot
plt.tight_layout()
plt.show()